# DiscoveryStack SEO/GEO 500：優化多任務訓練

本 Notebook 使用 owner-private `manifest-v4-500`，保留 250 筆 legacy regression 資料，新增 250 筆 web.dev 公開文件衍生樣本。`journeyStage` 使用 text pooled representation 加上 inference-safe `stageCueVector` branch；訓練選模只看 validation macro-F1，並執行 3 seeds、class-weight ablation、early stopping、zero-prediction gate 與 legacy test regression。

新增樣本仍是 development candidate；在 production 前必須完成 stage evidence 的人工 adjudication。原始 JSONL 不會進入 artifact ZIP。


In [ ]:
# Recovery environment: install only missing packages, then prove the runtime is usable.
import importlib.util, subprocess, sys
required = {'transformers': 'transformers', 'sklearn': 'scikit-learn', 'sentencepiece': 'sentencepiece', 'safetensors': 'safetensors'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print({'dependency_check': True, 'installed_missing': missing}, flush=True)
!nvidia-smi

from pathlib import Path
import hashlib, json, math, os, random, re, shutil, time, zipfile
from collections import Counter
from copy import deepcopy
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

EXPECTED_ROW_COUNT = 500
EXPECTED_MANIFEST_HASH = 'd1868ebd13ebf5b489e551afba2c047c19af5022e83c101e23a9927beaf02977'
EXPECTED_DATASET_DIGEST = '6aaf9e6c57f4d930ba220575bc1a5f7cb0ba6145373e7a8b629f89403c85c474'
EXPECTED_SPLITS = {'train': 350, 'validation': 75, 'test_legacy_v1': 32, 'test_v2': 43}
MODEL_ID = 'distilbert-base-multilingual-cased'
MODEL_VERSION = 'seo-geo-multitask-colab-v4-optimized'
FEATURE_CONTRACT_VERSION = 'features-v1'
TASKS = ['journeyStage','searchIntents','contentTypes','audienceRoles','geoSignals','citationReadiness','technicalSeoSignals','frictionSignals','actionPriority']
SINGLE_LABEL_TASKS = ['journeyStage','actionPriority']
MULTI_LABEL_TASKS = [task for task in TASKS if task not in SINGLE_LABEL_TASKS]
SEEDS = [20260820, 20260821, 20260822]
MAX_LENGTH, BATCH_SIZE, MAX_EPOCHS = 256, 8, 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'FAIL-CLOSED: CUDA GPU is required'
print({'imports_ready': True, 'model': MODEL_ID, 'device': str(DEVICE), 'rows': EXPECTED_ROW_COUNT, 'seeds': SEEDS}, flush=True)


In [ ]:
# Owner-private Drive load. The exact file name and digest are fail-closed.
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from io import FileIO

DRIVE_SNAPSHOT_NAME = 'discoverystack-manifest-v4-500-private.jsonl'
DATA_PATH = Path('/content/discoverystack-manifest-v4-500.jsonl')
auth.authenticate_user()
drive_service = build('drive', 'v3')
matches = drive_service.files().list(q=f"name = '{DRIVE_SNAPSHOT_NAME}' and trashed = false", spaces='drive', fields='files(id,name,size,trashed)').execute().get('files', [])
assert len(matches) == 1, f'FAIL-CLOSED: expected one private v4 snapshot, found {len(matches)}'
request = drive_service.files().get_media(fileId=matches[0]['id'])
with FileIO(DATA_PATH, 'wb') as target:
    downloader = MediaIoBaseDownload(target, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
assert DATA_PATH.stat().st_size > 100_000
print({'sourceName': matches[0]['name'], 'bytes': DATA_PATH.stat().st_size, 'private': True})


In [ ]:
# Fail-closed dataset, governance, split, and stage-balance validation.
raw_lines = [line.rstrip('\n') for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
rows = [json.loads(line) for line in raw_lines]
assert len(rows) == EXPECTED_ROW_COUNT
assert hashlib.sha256('\n'.join(raw_lines).encode()).hexdigest() == EXPECTED_DATASET_DIGEST
assert {row['manifestHash'] for row in rows} == {EXPECTED_MANIFEST_HASH}
assert len({int(row['id']) for row in rows}) == EXPECTED_ROW_COUNT
assert Counter(row['split'] for row in rows) == Counter(EXPECTED_SPLITS)
assert min(Counter(row['targets']['journeyStage'] for row in rows).values()) >= 80
for row in rows:
    assert row['reviewState'] in {'reviewed', 'needs_adjudication'}
    assert row['governance']['rightsStatus'] == 'approved'
    assert row['governance']['robotsChecked'] is True
    assert row['governance']['piiStatus'] in {'none_detected', 'masked'}
    assert row['stageEvidence']
print({'validated': True, 'rows': len(rows), 'splits': dict(Counter(row['split'] for row in rows)), 'stages': dict(Counter(row['targets']['journeyStage'] for row in rows))})


In [ ]:
# Label maps and inference-safe text cue features. No target/evidence fields are used here.
label_maps = {}
for task in TASKS:
    values = []
    for row in rows:
        value = row['targets'][task]
        values.extend([str(value)] if task in SINGLE_LABEL_TASKS else [str(item) for item in value])
    label_maps[task] = {label: i for i, label in enumerate(sorted(set(values)))}

CUE_PATTERNS = {
    'problem_statement_count': r'\b(problem|challenge|why|issue)\b',
    'question_heading_count': r'\?',
    'definition_cue_count': r'\b(what is|definition|means|introduction)\b',
    'how_it_works_cue_count': r'\b(how it works|steps|guide|tutorial|how to)\b',
    'comparison_cue_count': r'\b(compare|comparison|versus|pros|cons|alternative)\b',
    'requirements_cue_count': r'\b(requirement|prerequisite|eligibility)\b',
    'troubleshooting_cue_count': r'\b(troubleshoot|troubleshooting|fix|resolve)\b',
    'error_debug_cue_count': r'\b(error|debug|incorrect|failure)\b',
    'remediation_cue_count': r'\b(remediation|repair|improve)\b',
    'cta_count': r'\b(contact|book|buy|sign up|get started|request)\b',
    'contact_purchase_cue_count': r'\b(contact|purchase|order|checkout|shipping|returns)\b',
}
FEATURE_NAMES = list(CUE_PATTERNS) + ['form_presence', 'text_length_log', 'heading_count_log']

def raw_stage_features(row):
    text = row['trainingText'].lower()
    vector = [float(len(re.findall(pattern, text))) for pattern in CUE_PATTERNS.values()]
    vector.extend([float(bool(re.search(r'\b(form|email|phone)\b', text))), math.log1p(len(text)), math.log1p(text.count('\n'))])
    return np.asarray(vector, dtype=np.float32)

all_features = np.vstack([raw_stage_features(row) for row in rows])
train_idx = [i for i, row in enumerate(rows) if row['split'] == 'train']
feature_mean = all_features[train_idx].mean(axis=0)
feature_std = all_features[train_idx].std(axis=0)
feature_std[feature_std < 1e-6] = 1.0
all_features = (all_features - feature_mean) / feature_std
print({'featureContractVersion': FEATURE_CONTRACT_VERSION, 'featureNames': FEATURE_NAMES, 'featureDim': len(FEATURE_NAMES)})


In [ ]:
# Tokenization and tensors.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

class MultiTaskDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, position):
        index = self.indices[position]
        row = rows[index]
        enc = tokenizer(row['trainingText'], truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0), 'stage_features': torch.tensor(all_features[index], dtype=torch.float32)}
        for task in TASKS:
            value = row['targets'][task]
            if task in SINGLE_LABEL_TASKS:
                item[task] = torch.tensor(label_maps[task][str(value)], dtype=torch.long)
            else:
                vector = torch.zeros(len(label_maps[task]), dtype=torch.float32)
                for label in value: vector[label_maps[task][str(label)]] = 1.0
                item[task] = vector
        return item

indices_by_split = {split: [i for i, row in enumerate(rows) if row['split'] == split] for split in EXPECTED_SPLITS}
train_counts = Counter(str(rows[i]['targets']['journeyStage']) for i in indices_by_split['train'])
stage_order = list(label_maps['journeyStage'])
stage_weights = torch.tensor([(1.0 / max(1, train_counts[name])) ** 0.5 for name in stage_order], dtype=torch.float32, device=DEVICE)
stage_weights = stage_weights / stage_weights.mean()
print({'datasetSizes': {k: len(v) for k, v in indices_by_split.items()}, 'stageOrder': stage_order, 'stageWeights': stage_weights.detach().cpu().tolist()})


In [ ]:
class MultiTaskModelV4(nn.Module):
    def __init__(self, model_id, maps, feature_dim, use_stage_features=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden = self.encoder.config.hidden_size
        self.use_stage_features = use_stage_features
        branch_dim = 64 if use_stage_features else 0
        self.stage_feature_mlp = nn.Sequential(nn.LayerNorm(feature_dim), nn.Linear(feature_dim, 64), nn.GELU(), nn.Dropout(0.1)) if use_stage_features else None
        self.heads = nn.ModuleDict({task: nn.Linear(hidden + (branch_dim if task == 'journeyStage' else 0), len(maps[task])) for task in maps})
    def forward(self, input_ids, attention_mask, stage_features):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        stage_input = torch.cat([pooled, self.stage_feature_mlp(stage_features)], dim=-1) if self.use_stage_features else pooled
        return {task: self.heads[task](stage_input if task == 'journeyStage' else pooled) for task in self.heads}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def batch_to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


## Recovery gate: model definition and one-batch smoke test

這個 cell 必須在正式訓練前成功。它會重新建立一個模型、取一個 training batch，完成 forward、loss 與 backward，並輸出 `model_definition_ready` 與 `smoke_train_batch_ready`。若任一輸出缺失，停止，不要執行正式訓練。


In [ ]:
# One-batch smoke test: this is intentionally before train_one/full ablation.
smoke_model = MultiTaskModelV4(MODEL_ID, label_maps, len(FEATURE_NAMES), use_stage_features=True).to(DEVICE)
print({'model_definition_ready': True, 'device': str(DEVICE), 'featureDim': len(FEATURE_NAMES)}, flush=True)
smoke_loader = DataLoader(MultiTaskDataset(indices_by_split['train'][:8]), batch_size=min(BATCH_SIZE, 8), shuffle=False)
smoke_batch = batch_to_device(next(iter(smoke_loader)))
smoke_logits = smoke_model(smoke_batch['input_ids'], smoke_batch['attention_mask'], smoke_batch['stage_features'])
smoke_loss = 2.0 * nn.functional.cross_entropy(smoke_logits['journeyStage'], smoke_batch['journeyStage'])
for _task in MULTI_LABEL_TASKS:
    smoke_loss = smoke_loss + nn.functional.binary_cross_entropy_with_logits(smoke_logits[_task], smoke_batch[_task])
smoke_loss = smoke_loss + nn.functional.cross_entropy(smoke_logits['actionPriority'], smoke_batch['actionPriority'])
smoke_loss.backward()
print({'smoke_train_batch_ready': True, 'batchSize': int(smoke_batch['input_ids'].shape[0]), 'seqLength': int(smoke_batch['input_ids'].shape[1]), 'lossFinite': bool(torch.isfinite(smoke_loss).item())}, flush=True)
del smoke_model, smoke_loader, smoke_batch, smoke_logits, smoke_loss
torch.cuda.empty_cache()


## Formal training: truthful run mode

預設為 `fast_path`，只跑 `stage_branch_weighted`、seed `20260820`、最多 2 epochs；這是可觀察的快速候選，不等同原定 2 configs × 3 seeds 的完整 ablation。若 T4、smoke test 與第一個 run 都穩定，將本 cell 的 `RUN_MODE` 改為 `full_ablation` 後由新鮮 runtime 重新執行，才可產生完整比較結果。所有選模仍只使用 validation `journeyStage` macro-F1，`test_v2` 不得用於調參。


In [ ]:
def evaluate(model, indices):
    model.eval()
    loader = DataLoader(MultiTaskDataset(indices), batch_size=BATCH_SIZE, shuffle=False)
    truths, predictions = {task: [] for task in TASKS}, {task: [] for task in TASKS}
    total_loss = 0.0; batches = 0
    ce = nn.CrossEntropyLoss(weight=stage_weights)
    bce = {task: nn.BCEWithLogitsLoss() for task in MULTI_LABEL_TASKS}
    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch); logits = model(batch['input_ids'], batch['attention_mask'], batch['stage_features'])
            loss = ce(logits['journeyStage'], batch['journeyStage'])
            for task in MULTI_LABEL_TASKS: loss = loss + bce[task](logits[task], batch[task])
            loss = loss + nn.functional.cross_entropy(logits['actionPriority'], batch['actionPriority'])
            total_loss += float(loss.item()); batches += 1
            for task in SINGLE_LABEL_TASKS:
                truths[task].extend(batch[task].detach().cpu().tolist()); predictions[task].extend(logits[task].argmax(-1).detach().cpu().tolist())
            for task in MULTI_LABEL_TASKS:
                truths[task].extend(batch[task].detach().cpu().numpy().tolist()); predictions[task].extend((torch.sigmoid(logits[task]) >= 0.5).detach().cpu().numpy().tolist())
    stage_true, stage_pred = truths['journeyStage'], predictions['journeyStage']
    stage_macro = float(f1_score(stage_true, stage_pred, average='macro', zero_division=0))
    stage_micro = float(f1_score(stage_true, stage_pred, average='micro', zero_division=0))
    per_class = f1_score(stage_true, stage_pred, labels=list(range(len(stage_order))), average=None, zero_division=0).tolist()
    matrix = confusion_matrix(stage_true, stage_pred, labels=list(range(len(stage_order)))).tolist()
    predicted_support = Counter(stage_pred)
    return {'loss': total_loss / max(1, batches), 'macroF1': stage_macro, 'microF1': stage_micro, 'accuracy': float(accuracy_score(stage_true, stage_pred)), 'perClassF1': dict(zip(stage_order, per_class)), 'confusionMatrix': matrix, 'predictedSupport': {stage_order[i]: int(predicted_support.get(i, 0)) for i in range(len(stage_order))}, 'truths': truths, 'predictions': predictions}

def train_one(seed, config_name, use_stage_features, use_class_weight):
    set_seed(seed)
    print({'run_start': True, 'config': config_name, 'seed': seed, 'maxEpochs': TRAIN_MAX_EPOCHS}, flush=True)
    model = MultiTaskModelV4(MODEL_ID, label_maps, len(FEATURE_NAMES), use_stage_features=use_stage_features).to(DEVICE)
    loader = DataLoader(MultiTaskDataset(indices_by_split['train']), batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    ce = nn.CrossEntropyLoss(weight=stage_weights if use_class_weight else None)
    bce = {task: nn.BCEWithLogitsLoss() for task in MULTI_LABEL_TASKS}
    best_score, best_epoch, best_state, patience = -1.0, 0, None, 0
    history = []
    for epoch in range(1, TRAIN_MAX_EPOCHS + 1):
        model.train(); total = 0.0
        print({'epoch_start': True, 'config': config_name, 'seed': seed, 'epoch': epoch}, flush=True)
        for batch in loader:
            batch = batch_to_device(batch); optimizer.zero_grad(set_to_none=True)
            logits = model(batch['input_ids'], batch['attention_mask'], batch['stage_features'])
            loss = 2.0 * ce(logits['journeyStage'], batch['journeyStage'])
            for task in MULTI_LABEL_TASKS: loss = loss + bce[task](logits[task], batch[task])
            loss = loss + nn.functional.cross_entropy(logits['actionPriority'], batch['actionPriority'])
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); total += float(loss.item())
        val = evaluate(model, indices_by_split['validation'])
        record = {'epoch': epoch, 'trainLoss': total / max(1, len(loader)), 'validation': {k: v for k, v in val.items() if k not in {'truths','predictions'}}}
        history.append(record); print({'epoch_complete': True, 'config': config_name, 'seed': seed, 'epoch': epoch, 'trainLoss': record['trainLoss'], 'valMacroF1': val['macroF1'], 'valPredictedSupport': val['predictedSupport']}, flush=True)
        if val['macroF1'] > best_score + 1e-5:
            best_score, best_epoch, best_state, patience = val['macroF1'], epoch, deepcopy(model.state_dict()), 0
        else:
            patience += 1
            if patience >= 2: break
    model.load_state_dict(best_state); best_val = evaluate(model, indices_by_split['validation'])
    return model, best_val, {'config': config_name, 'seed': seed, 'bestEpoch': best_epoch, 'history': history, 'validation': {k: v for k, v in best_val.items() if k not in {'truths','predictions'}}}

# Default is a bounded, observable fast path. Set RUN_MODE='full_ablation' only after the smoke gate passes.
RUN_MODE = 'fast_path'  # 'fast_path' = one weighted branch / one seed / max 2 epochs; 'full_ablation' = 2 configs x 3 seeds x 8 epochs
if RUN_MODE == 'full_ablation':
    configs = [('text_only_baseline', False, False), ('stage_branch_weighted', True, True)]
    SEEDS_TO_RUN = SEEDS
    TRAIN_MAX_EPOCHS = MAX_EPOCHS
else:
    configs = [('stage_branch_weighted', True, True)]
    SEEDS_TO_RUN = [SEEDS[0]]
    TRAIN_MAX_EPOCHS = 2
print({'training_mode': RUN_MODE, 'configs': configs, 'seeds': SEEDS_TO_RUN, 'maxEpochs': TRAIN_MAX_EPOCHS}, flush=True)
run_root = Path('/content/optimized_runs_v4_recovery'); shutil.rmtree(run_root, ignore_errors=True); run_root.mkdir(parents=True)
run_records = []; run_paths = []
for config_name, use_stage_features, use_class_weight in configs:
    for seed in SEEDS_TO_RUN:
        model, val, record = train_one(seed, config_name, use_stage_features, use_class_weight)
        path = run_root / f'{config_name}__seed_{seed}.pt'
        torch.save({'state_dict': model.state_dict(), 'config': record, 'useStageFeatures': use_stage_features, 'useClassWeight': use_class_weight}, path)
        run_records.append(record); run_paths.append(str(path))
        del model; torch.cuda.empty_cache()
best_index = max(range(len(run_records)), key=lambda i: run_records[i]['validation']['macroF1'])
selected = run_records[best_index]; selected_path = Path(run_paths[best_index])
print({'selected': selected, 'candidateCount': len(run_records), 'training_complete': True, 'training_mode': RUN_MODE}, flush=True)


In [ ]:
# Final selected model evaluation: test_v2 is new test; test_legacy_v1 is fixed regression only.
checkpoint = torch.load(selected_path, map_location=DEVICE)
selected_config = checkpoint['config']['config']
use_stage_features = bool(checkpoint['useStageFeatures'])
final_model = MultiTaskModelV4(MODEL_ID, label_maps, len(FEATURE_NAMES), use_stage_features=use_stage_features).to(DEVICE)
final_model.load_state_dict(checkpoint['state_dict'])
test_v2 = evaluate(final_model, indices_by_split['test_v2'])
test_legacy = evaluate(final_model, indices_by_split['test_legacy_v1'])
zero_prediction_classes = [name for name, support in test_v2['predictedSupport'].items() if support == 0]
readiness = 'candidate_not_ready' if zero_prediction_classes else 'candidate_ready_for_review'
metrics = {'modelVersion': MODEL_VERSION, 'selectedConfig': selected_config, 'selectedSeed': selected['seed'], 'selectedValidation': selected['validation'], 'test_v2': {k:v for k,v in test_v2.items() if k not in {'truths','predictions'}}, 'test_legacy_v1': {k:v for k,v in test_legacy.items() if k not in {'truths','predictions'}}, 'zeroPredictionClasses': zero_prediction_classes, 'readiness': readiness, 'runRecords': run_records}
print(json.dumps(metrics, ensure_ascii=False, indent=2))


In [ ]:
# Allow-list artifact packaging. Raw JSONL, HTML, browser state, and secrets are excluded.
artifact_root = Path('/content/discoverystack-training-artifacts-v4'); shutil.rmtree(artifact_root, ignore_errors=True); artifact_root.mkdir(parents=True)
model_dir = artifact_root / 'model'; model_dir.mkdir()
torch.save({'state_dict': final_model.state_dict(), 'label_maps': label_maps, 'modelId': MODEL_ID, 'modelVersion': MODEL_VERSION, 'useStageFeatures': use_stage_features}, model_dir / 'model_state_dict.pt')
tokenizer.save_pretrained(str(model_dir / 'tokenizer'))
(artifact_root / 'training_config.json').write_text(json.dumps({'modelId':MODEL_ID,'modelVersion':MODEL_VERSION,'featureContractVersion':FEATURE_CONTRACT_VERSION,'maxLength':MAX_LENGTH,'batchSize':BATCH_SIZE,'maxEpochs':MAX_EPOCHS,'seeds':SEEDS,'selectedConfig':selected_config,'selectedSeed':selected['seed'],'stageLossWeight':2.0}, ensure_ascii=False, indent=2), encoding='utf-8')
(artifact_root / 'label_maps.json').write_text(json.dumps(label_maps, ensure_ascii=False, indent=2), encoding='utf-8')
(artifact_root / 'feature_stats.json').write_text(json.dumps({'featureNames':FEATURE_NAMES,'mean':feature_mean.tolist(),'std':feature_std.tolist(),'contractVersion':FEATURE_CONTRACT_VERSION}, ensure_ascii=False, indent=2), encoding='utf-8')
(artifact_root / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
for split_name, result in [('test_v2', test_v2), ('test_legacy_v1', test_legacy)]:
    with (artifact_root / f'{split_name}_predictions.jsonl').open('w', encoding='utf-8') as handle:
        for position, row_index in enumerate(indices_by_split[split_name]):
            handle.write(json.dumps({'id': int(rows[row_index]['id']), 'split': split_name, 'true': {task: result['truths'][task][position] for task in TASKS}, 'pred': {task: result['predictions'][task][position] for task in TASKS}}, ensure_ascii=False) + '\n')

def file_sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
files = {}
for path in sorted(artifact_root.rglob('*')):
    if path.is_file(): files[str(path.relative_to(artifact_root))] = {'bytes': path.stat().st_size, 'sha256': file_sha(path)}
artifact_manifest = {'artifactVersion':'artifact-v4-500-optimized','manifestHash':EXPECTED_MANIFEST_HASH,'datasetDigest':EXPECTED_DATASET_DIGEST,'modelVersion':MODEL_VERSION,'readiness':readiness,'containsRawDataset':False,'containsHtml':False,'files':files}
(artifact_root / 'artifact-manifest.json').write_text(json.dumps(artifact_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
zip_path = Path('/content') / f'discoverystack-ml-v4-500-{EXPECTED_MANIFEST_HASH[:12]}.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(artifact_root.rglob('*')):
        if path.is_file(): archive.write(path, path.relative_to(artifact_root))
zip_sha256 = file_sha(zip_path)
print({'artifactRoot':str(artifact_root),'zip':str(zip_path),'zipBytes':zip_path.stat().st_size,'zipSha256':zip_sha256,'containsRawDataset':False,'readiness':readiness})


In [ ]:
# Save the optimized artifact ZIP to owner-only Drive.
from googleapiclient.http import MediaFileUpload
zip_name = zip_path.name
existing = drive_service.files().list(q=f"name = '{zip_name}' and trashed = false", spaces='drive', fields='files(id,name,size,webViewLink)').execute().get('files', [])
if existing:
    drive_artifact = existing[0]
else:
    drive_artifact = drive_service.files().create(body={'name':zip_name,'description':'Owner-only DiscoveryStack v4 optimized 500-row artifact; raw dataset and HTML excluded'}, media_body=MediaFileUpload(str(zip_path), mimetype='application/zip', resumable=True), fields='id,name,size,webViewLink').execute()
print({'privateDriveArtifact':True,'file':drive_artifact,'zipSha256':zip_sha256,'rawDataIncluded':False,'readiness':readiness})


## Gate interpretation

`test_v2` is evaluated only after validation-based selection across the ablation and seed grid. `test_legacy_v1` is a regression check and must remain fixed. If any journeyStage class has zero predicted support, the artifact is marked `candidate_not_ready`; this is a failure signal to improve data or labels, not a reason to change the threshold or claim success.
